08 Prompt Engineering with LangChain and ChatGPT.ipynb

Project: Prompt Engineering with LangChain and ChatGPT for real-world tasks

In this notebook you will leverage ChatGPT and LangChain to solve and do a few mini-projects based on some real-world scenarios:

Mini-Project 1: Review Analyst
Mini-Project 2: Research Paper Analyst
Mini-Project 3: Social Media Marketing Analyst
Mini-Project 4: IT Support Analyst

Mini-Project 1: Review Analyst

You are building an AI system to be able to look at customer reviews and do some complex analysis. for each review get ChatGPT to do the following:

Summarize the review. The summary should be at most 3 lines.
Highlight both the positives and negatives
Display the overall sentiment of the review (positive, negative, neutral)
Display a list of 3 - 5 emotions expressed by the customer in the review
If the sentiment is positive or neutral write an email and thank them for the review
If the sentiment is negative apologize and write an email with an appropriate response
Try to get the response in a nice structured format using an output parser

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [3]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


Load Necessary Dependencies and ChatGPT LLM

In [5]:
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

In [6]:
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

In [7]:
reviews = [
    f"""
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    """,
    f"""
    Purchased a new gaming keyboard because of its rave reviews about responsiveness and backlighting.
    It hasn't disappointed. The keys have a satisfying click and the LED colors are vibrant,
    enhancing my gaming experience significantly. Price-wise, it's quite competitive,
    and I feel like I got a good deal. The delivery was swift, and it came well-protected,
    ensuring no damage during transport.
    """,
    f"""
    Ordered a set of wireless earbuds for running, and they've been a letdown.
    The sound constantly cuts out, and the fit is uncomfortable after only a few minutes of use.
    They advertised a 12-hour battery life, but I'm barely getting four hours.
    Considering the cost, I expected better quality and performance.
    They did arrive on time, but the positives end there. I'm already looking into a return.
    """,
    f"""
    The tablet stand I bought was touted as being sturdy and adjustable,
    but it's anything but. It wobbles with the slightest touch,
    and the angles are not holding up as promised. It feels like a breeze could knock it over.
    It was also pricier than others I've seen, which adds to the disappointment.
    It did arrive promptly, but what's the use if the product doesn't meet basic expectations?
    """,
    f"""
    Needed a new kitchen blender, but this model has been a nightmare.
    It's supposed to handle various foods, but it struggles with anything tougher than cooked vegetables.
    It's also incredibly noisy, and the 'easy-clean' feature is a joke; food gets stuck under the blades constantly.
    I thought the brand meant quality, but this product has proven me wrong.
    Plus, it arrived three days late. Definitely not worth the expense.
    """
]

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [10]:
# Define your desired data structure - like a python data class.
class ReviewAnalysisResponse(BaseModel):
    summary: str =   Field(description = "A brief summary of the customer review with maximum 3 lines")
    positives: str = Field(description = "A list showing the positives mentioned by the customer in the review if any - max 3 points")
    negatives: str = Field(description = "A list showing the negatives mentioned by the customer in the review if any - max 3 points")
    sentiment: str = Field(description = "One word showing the sentiment of the review - positive, negative or neutral")
    emotions: str =  Field(description = "A list of 3 - 5 emotions expressed by the customer in the review")
    email: str =     Field(description = "Detailed email to the customer based on the sentiment")

# Set up a parser + inject instructions into the prompt template.
parser = PydanticOutputParser(pydantic_object=ReviewAnalysisResponse)
parser
    
    

PydanticOutputParser(pydantic_object=<class '__main__.ReviewAnalysisResponse'>)

In [11]:
# create the final prompt with formatting instructions from the parser
prompt_txt = """
             Analyze the given customer review below and generate the response based on the instructions
             mentioned below in the format instructions.
             Also remember to write a detailed email response for the email field based on these conditions:
               - email should be addressed to Dear Customer and signed with Service Agent
               - thank them if the review is positive or neutral
               - apologize if the review is negative

             Format Instructions:
             {format_instructions}

             Review:
             {review}
            """
prompt = PromptTemplate(template=prompt_txt,input_variables=["review"],partial_variables={"format_instructions": parser.get_format_instructions()},)

In [14]:
prompt

PromptTemplate(input_variables=['review'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"summary": {"description": "A brief summary of the customer review with maximum 3 lines", "title": "Summary", "type": "string"}, "positives": {"description": "A list showing the positives mentioned by the customer in the review if any - max 3 points", "title": "Positives", "type": "string"}, "negatives": {"description": "A list showing the negatives mentioned by the customer in the review if any - max 3 points", "title":

Create a LCEL LLM Chain

In [15]:
# create a simple LCEL chain to take the prompt, pass it to the LLM, enforce response format using the parser
chain = (prompt 
           |
        llm
           |
        parser)

In [18]:
review_formatted = [{'review':review} for review in reviews]
review_formatted[0]

{'review': "\n    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.\n    The sound quality is impressively clear with just the right amount of bass.\n    It's also waterproof, which tested true during a recent splashing incident.\n    Though it's compact, the volume can really fill the space.\n    The price was a bargain for such high-quality sound.\n    Shipping was also on point, arriving two days early in secure packaging.\n    "}

In [20]:
response = chain.map().invoke(review_formatted)

In [21]:
response[0]

ReviewAnalysisResponse(summary='The customer is very pleased with the Bluetooth speaker, highlighting its sound quality, waterproof feature, and value for money.', positives='Fantastic sound quality, waterproof, great price.', negatives='', sentiment='positive', emotions='satisfaction, excitement, appreciation', email="Dear Customer,\n\nThank you for your wonderful review! We're thrilled to hear that you are enjoying your new Bluetooth speaker and that it met your expectations in sound quality and waterproof capabilities. It's great to know that you found it to be a fantastic value as well. We appreciate your feedback and hope you have many enjoyable beach outings with your speaker!\n\nBest regards,\nService Agent")

In [23]:
response[0].dict()

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_17622/15774499.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response[0].dict()


{'summary': 'The customer is very pleased with the Bluetooth speaker, highlighting its sound quality, waterproof feature, and value for money.',
 'positives': 'Fantastic sound quality, waterproof, great price.',
 'negatives': '',
 'sentiment': 'positive',
 'emotions': 'satisfaction, excitement, appreciation',
 'email': "Dear Customer,\n\nThank you for your wonderful review! We're thrilled to hear that you are enjoying your new Bluetooth speaker and that it met your expectations in sound quality and waterproof capabilities. It's great to know that you found it to be a fantastic value as well. We appreciate your feedback and hope you have many enjoyable beach outings with your speaker!\n\nBest regards,\nService Agent"}

In [27]:
for resp in response:
    for k,v in resp.dict().items():
      print(f'{k}:\n{v}')
      print('-----')
      print('\n')

summary:
The customer is very pleased with the Bluetooth speaker, highlighting its sound quality, waterproof feature, and value for money.
-----


positives:
Fantastic sound quality, waterproof, great price.
-----


negatives:

-----


sentiment:
positive
-----


emotions:
satisfaction, excitement, appreciation
-----


email:
Dear Customer,

Thank you for your wonderful review! We're thrilled to hear that you are enjoying your new Bluetooth speaker and that it met your expectations in sound quality and waterproof capabilities. It's great to know that you found it to be a fantastic value as well. We appreciate your feedback and hope you have many enjoyable beach outings with your speaker!

Best regards,
Service Agent
-----


summary:
The customer is highly satisfied with their new gaming keyboard, praising its responsiveness, backlighting, and competitive pricing.
-----


positives:
Responsive keys, vibrant LED colors, competitive price.
-----


negatives:

-----


sentiment:
positive
-

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_17622/3121286396.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for k,v in resp.dict().items():


In [28]:
import pandas as pd

In [29]:
pd.DataFrame(resp.dict() for resp in response)

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_17622/1782428282.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  pd.DataFrame(resp.dict() for resp in response)


,summary,positives,negatives,sentiment,emotions,email
0,The customer is very pleased with the Bluetoot...,"Fantastic sound quality, waterproof, great price.",,positive,"satisfaction, excitement, appreciation","Dear Customer,\n\nThank you for your wonderful..."
1,The customer is highly satisfied with their ne...,"Responsive keys, vibrant LED colors, competiti...",,positive,"satisfaction, excitement, happiness","Dear Customer,\n\nThank you for your wonderful..."
2,The customer is disappointed with the wireless...,The earbuds arrived on time.,"Sound cuts out, uncomfortable fit, poor batter...",negative,"disappointment, frustration, dissatisfaction","Dear Customer,\n\nThank you for your feedback ..."
3,The customer is disappointed with the tablet s...,The product arrived promptly.,"The stand wobbles easily, angles do not hold, ...",negative,"disappointment, frustration, dissatisfaction","Dear Customer,\n\nThank you for your feedback ..."
4,The customer is dissatisfied with the kitchen ...,,"Struggles with tough foods, very noisy, ineffe...",negative,"frustration, disappointment, dissatisfaction, ...","Dear Customer,\n\nThank you for taking the tim..."
